# 🎯 실습: Tool Calling

 `chain = prompt | llm | parser` → LLM이 **자기 머릿속 정보로만** 답변  
 `llm.bind_tools([...])` → LLM이 **외부 도구를 호출**해서 답변  

체인에 부품을 더하는 게 아니라, **LLM 자체에 능력을 부여**합니다.

## 📋 빌드업 흐름

```
[1-3교시] chain = prompt | llm | parser
              ↓ LLM의 한계 체험
[추가 ①] @tool 데코레이터로 도구 정의
[추가 ②] bind_tools로 LLM에 도구 등록
[추가 ③] tool_calls — LLM의 도구 호출 의도 읽기
[추가 ④] 도구 실행 → LLM에 결과 전달 → 최종 답변
[추가 ⑤] ⭐ 도구 반환값도 Pydantic 정형화
   1-2교시의 Pydantic 패턴이 도구에도 그대로
```

## 📋 노트북 순서

```
Step 0. 환경 설정
Step 1. LLM의 한계 직접 체험
Step 2. ① @tool 첫 도구 만들기
Step 3. ② bind_tools — LLM에 도구 등록
Step 4. ③ tool_calls — LLM의 의도 읽기
Step 5. ④ 도구 실행 + 2차 호출 = 완전한 흐름
Step 6. 헬퍼 함수로 묶기 ⭐
Step 7. ⭐ Pydantic 정형화 + 5가지 실무 미니 케이스
   7-0. 도구 반환값을 Pydantic으로 (핵심 패턴)
   7-1. 환율 변환기 → ExchangeRateResult
   7-2. 현재 시각·날짜 → TimeInfo
   7-3. 상품 검색 → ProductInfo (1-2교시 패턴 재사용)
   7-4. 여러 도구 한 번에 등록
   7-5. 도구를 안 써도 되는 질문
Step 8. 도전 과제
```




---

# Step 0. 환경 설정


In [1]:
# !uv add langchain langchain-openai python-dotenv pydantic

## 공통 import

이전 시간 도구 + 오늘 새로 배울 도구.


In [5]:
from typing import Literal, Optional
from pydantic import BaseModel, Field   
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool   # ⭐ 오늘의 주인공

# Tool Calling은 temperature=0 권장 (도구 선택의 일관성)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ import 완료


---

# Step 1. LLM의 한계 직접 체험

오늘 새 기법이 왜 필요한지 **직접 보는** 게 먼저예요. 다음 3가지 질문을 평범한 LLM에게 던져봅시다.


In [3]:
# LLM이 모르는 3가지 질문
questions = [
    "지금 몇 시야?",
    "오늘 USD/KRW 환율 알려줘",
    "우리 쇼핑몰에서 'iPhone 15' 재고 몇 개야?",
]

for q in questions:
    print(f"❓ {q}")
    print(f"💬 {llm.invoke(q).content[:120]}...")
    print()

❓ 지금 몇 시야?
💬 죄송하지만, 현재 시간을 확인할 수 있는 기능은 없습니다. 사용 중인 기기나 시계를 통해 시간을 확인해 보시기 바랍니다. 도움이 필요하시면 다른 질문 해주세요!...

❓ 오늘 USD/KRW 환율 알려줘
💬 죄송하지만, 실시간 환율 정보를 제공할 수는 없습니다. USD/KRW 환율을 확인하시려면 금융 뉴스 웹사이트나 환율 관련 앱을 참고하시기 바랍니다....

❓ 우리 쇼핑몰에서 'iPhone 15' 재고 몇 개야?
💬 죄송하지만, 저는 실시간 데이터에 접근할 수 없기 때문에 특정 쇼핑몰의 재고 정보를 확인할 수 없습니다. 해당 쇼핑몰의 웹사이트나 고객 서비스에 문의하시면 정확한 재고 정보를 얻으실 수 있습니다. 도움이 필요하시면 ...



**👀 결과 관찰**

LLM이 어떻게 답하나요? 보통 이렇게 말해요.

- "저는 실시간 정보에 접근할 수 없어요"
- "환율은 매일 변하기 때문에 알 수 없습니다"
- "쇼핑몰 데이터베이스에 접근 권한이 없어요"

**LLM이 못 하는 4가지**

| 종류 | 예시 |
| --- | --- |
| 실시간 정보 | 환율, 날씨, 주가 |
| 현재 상태 | 시각, 위치 |
| 정확한 계산 | 큰 수의 곱셈, 날짜 차이 |
| 외부 시스템 접근 | DB, API, 파일 |

→ 이걸 해결하는 게 **Tool Calling** 입니다. LLM에게 "이 일은 이 도구를 쓰면 돼" 하고 알려주는 거예요.


---

# Step 2. ① `@tool` 데코레이터로 첫 도구 만들기

가장 단순한 도구부터 만들어봅시다. **함수 위에 `@tool` 한 줄** 붙이면 끝.

## 2-1. 곱셈 도구 정의

LLM은 큰 수 곱셈을 잘 못해요. 그래서 곱셈 도구를 만들어줍니다.


In [16]:
# 가장 단순한 도구
@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱한 결과를 반환합니다.

    곱셈 계산이 필요할 때 사용하세요.
    예: 1234 * 5678, 100 * 1.5
    """
    return a * b

print("✅ 첫 도구 정의 완료")

✅ 첫 도구 정의 완료


**👀 4가지 핵심 요소**

| 요소 | 위 코드에서 |
| --- | --- |
| 함수명 | `multiply` |
| docstring | "두 수를 곱한 결과를 반환합니다..." |
| 타입 힌트 | `a: float, b: float` |
| 반환값 | `a * b` |

이 4가지를 LLM이 읽고 **"언제 이 도구를 써야 하는지"** 판단해요.

## 2-2. 도구 정보 확인


In [5]:
print(f"📛 이름:    {multiply.name}")
print(f"📝 설명:    {multiply.description}")
print(f"🔧 인자:    {multiply.args}")

📛 이름:    multiply
📝 설명:    두 수를 곱한 결과를 반환합니다.

    곱셈 계산이 필요할 때 사용하세요.
    예: 1234 * 5678, 100 * 1.5
🔧 인자:    {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}


**👀 결과**

`@tool` 데코레이터가 함수를 분석해서 메타데이터를 자동 생성했어요. LLM은 이 정보만 보고 도구를 사용합니다.

→ **docstring과 타입 힌트가 LLM에게 보내는 사용 설명서**입니다.


---

# Step 3. ② `bind_tools` — LLM에 도구 등록

도구를 만들었으면 LLM에게 알려줘야 해요. **`bind_tools()`** 메서드를 씁니다.

```
llm                       ← 평범한 LLM
   ↓ .bind_tools([multiply])
llm_with_tools            ← 도구 사용 가능한 LLM ⭐
```


In [6]:
# LLM에 도구 등록
llm_with_tools = llm.bind_tools([multiply])

print("✅ multiply 도구를 LLM에 등록 완료")
print(f"   타입: {type(llm_with_tools).__name__}")

✅ multiply 도구를 LLM에 등록 완료
   타입: _ChatModelBinding


`llm_with_tools` 는 일반 LLM처럼 동작하지만 **필요할 때 도구를 호출**할 수 있어요. 어떻게 동작하는지 다음 단계에서 봅시다.


---

# Step 4. ③ `tool_calls` — LLM의 의도 읽기

도구가 등록된 LLM에게 두 종류의 질문을 던져봅시다.

## 4-1. 일반 질문 (도구 불필요)


In [7]:
# 도구가 필요 없는 질문
response1 = llm_with_tools.invoke("안녕! 너 누구야?")
print(f"💬 content:    {response1.content}")
print(f"🔧 tool_calls: {response1.tool_calls}")

💬 content:    안녕하세요! 저는 AI 언어 모델로, 다양한 질문에 답하고 정보를 제공하는 역할을 하고 있습니다. 무엇을 도와드릴까요?
🔧 tool_calls: []


**👀 결과**

- `content` 는 일반 답변
- `tool_calls` 는 빈 리스트 `[]` — 도구를 안 씀

## 4-2. 계산이 필요한 질문 (도구 사용)


In [8]:
# 곱셈이 필요한 질문
response2 = llm_with_tools.invoke("1234 곱하기 5678이 얼마야?")
print(f"💬 content:    '{response2.content}'")
print(f"🔧 tool_calls: {response2.tool_calls}")

💬 content:    ''
🔧 tool_calls: [{'name': 'multiply', 'args': {'a': 1234, 'b': 5678}, 'id': 'call_a8B8T0hWtBHXDJNANRACnqat', 'type': 'tool_call'}]


**👀 결과 관찰 — 매우 중요**

- `content` 가 **비어있어요!** `''`
- `tool_calls` 에 **도구 호출 정보**가 채워져 있어요

이게 LLM이 "내가 직접 답하는 게 아니라 multiply 도구를 써야 한다" 고 알려주는 신호예요.

## 4-3. tool_calls 구조 살펴보기


In [9]:
tc = response2.tool_calls[0]   # 첫 번째 도구 호출 정보
print(f"📛 어떤 도구를:  {tc['name']}")
print(f"🔢 어떤 인자로:  {tc['args']}")
print(f"🆔 호출 ID:      {tc['id']}")

📛 어떤 도구를:  multiply
🔢 어떤 인자로:  {'a': 1234, 'b': 5678}
🆔 호출 ID:      call_a8B8T0hWtBHXDJNANRACnqat


**LLM이 우리에게 말하는 것**

> "multiply 도구를 a=1234, b=5678 인자로 호출해줘. 호출 ID는 call_xxx야."

LLM은 **도구를 직접 실행하지 않아요**. 우리에게 "이 도구를 이렇게 써줘" 라고 요청만 합니다. 실제 실행은 우리가 합니다.


---

# Step 5. ④ 도구 실행 + 2차 호출 = 완전한 흐름

이제 진짜 답변까지 받아봅시다. **4단계 흐름** 전체를 한 번 따라갑니다.

```
[1단계] 사용자 질문        ← 우리가 작성
[2단계] LLM 1차 호출       ← LLM이 "도구 써줘" 응답 (Step 4)
[3단계] 도구 실행          ← 우리가 직접 호출
[4단계] LLM 2차 호출       ← 결과 보고 최종 답변 생성
```

## 5-1. 1·2단계: 메시지 구성 + 1차 호출


In [10]:
# 메시지 누적용 리스트
messages = [HumanMessage(content="1234 곱하기 5678이 얼마야?")]

# 1차 호출 — LLM이 도구 호출 의도를 응답
response = llm_with_tools.invoke(messages)
messages.append(response)

print(f"🔧 도구 호출 의도: {response.tool_calls}")

🔧 도구 호출 의도: [{'name': 'multiply', 'args': {'a': 1234, 'b': 5678}, 'id': 'call_mPNiIYC28vJKefoWq6kYlCQF', 'type': 'tool_call'}]


## 5-2. 3단계: 도구 실행

`tool_calls` 의 정보를 보고 직접 도구를 실행합니다. 결과는 **`ToolMessage`** 로 감싸서 메시지 흐름에 추가해요.


In [11]:
# 도구 호출 정보 꺼내기
tc = response.tool_calls[0]

# 실제 도구 실행 — .invoke()에 인자 dict 그대로 전달
result = multiply.invoke(tc["args"])
print(f"📊 도구 실행 결과: {result}")

# 결과를 ToolMessage로 감싸서 메시지에 추가
messages.append(ToolMessage(
    content=str(result),
    tool_call_id=tc["id"]
))

print(f"✅ 메시지 흐름에 도구 결과 추가됨 (총 {len(messages)}개)")

📊 도구 실행 결과: 7006652.0
✅ 메시지 흐름에 도구 결과 추가됨 (총 3개)


## 5-3. 4단계: 2차 호출 — 최종 답변


In [12]:
# 도구 결과를 본 LLM이 자연어 답변 생성
final = llm_with_tools.invoke(messages)
print(f"💬 최종 답변: {final.content}")

💬 최종 답변: 1234 곱하기 5678은 7,006,652입니다.


**🎉 첫 Tool Calling 성공!**

```
[0] 👤 HumanMessage:  사용자 질문
[1] 🤖 AIMessage:     "이 도구 써줘" (content는 비어있음)
[2] 🔧 ToolMessage:   도구 실행 결과
[3] 🤖 AIMessage:     최종 자연어 답변
```

**LLM이 두 번 호출**됐고, 그 사이에 **도구가 한 번 실행**됐습니다.


---

# Step 6. 헬퍼 함수로 묶기 ⭐

매번 4단계를 직접 짜면 번거롭죠. 위 흐름을 **함수 하나로 묶어두면**, 이후 케이스에서는 함수만 호출하면 됩니다.

> 💡 5교시에서 배울 ReAct Agent가 사실 이 흐름의 **자동화 버전**이에요.


In [6]:
def run_with_tools(question: str, tools: list, llm_obj=llm):
    """질문 + 도구 리스트를 받아 4단계 흐름을 실행하고 최종 답변 반환"""
    llm_with_tools = llm_obj.bind_tools(tools)
    messages = [HumanMessage(content=question)]

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
        return response.content

    tool_map = {t.name: t for t in tools}
    for tc in response.tool_calls:
        result = tool_map[tc["name"]].invoke(tc["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return llm_with_tools.invoke(messages).content

print("✅ 헬퍼 함수 준비 완료")

✅ 헬퍼 함수 준비 완료


In [ ]:
# Step 5의 4단계를 한 줄로
answer = run_with_tools("1234 곱하기 5678이 얼마야?", [multiply])
print(answer)

**🎉 한 줄로 끝!**

이제 다양한 도구와 질문을 손쉽게 테스트할 수 있어요. 다음 Part에서는 이 헬퍼 함수만 사용하면서 **Pydantic 정형화를 더한 5가지 실무 사례**를 봅니다.


---

# Step 7. ⭐ Pydantic 정형화 + 5가지 실무 미니 케이스

## 7-0. 도구 반환값을 Pydantic으로 — 핵심 패턴 ⭐

LLM 답변을 Pydantic으로 정형화했고, 함수 반환값도 Pydantic으로 받았어요. **오늘은 도구(`@tool`) 반환값도 Pydantic으로** 정형화합니다.

```python
# 기존 (Step 5 multiply)
@tool
def multiply(a: float, b: float) -> float:
    return a * b

# 개선: dict나 단순 타입 대신 Pydantic 객체 반환
class CalcResult(BaseModel):
    operation: str
    result: float

@tool
def multiply_v2(a: float, b: float) -> CalcResult:
    return CalcResult(operation=f"{a} × {b}", result=a * b)
```

**이렇게 하면 좋은 점**

| 효과 | 설명 |
| --- | --- |
| 1-2교시 패턴 일관성 | OutputParser·Pydantic 학습이 4교시에도 이어짐 |
| 자동 검증 | 잘못된 값이 들어가면 즉시 에러 (디버깅 ↑) |
| IDE 자동완성 | `result.operation`, `result.result` 속성 접근 가능 |
| 직렬화 자동 | LangChain이 LLM에 전달할 때 자동으로 JSON 변환 |

> 💡 **LLM 입장에서는 dict든 Pydantic이든 똑같이 받습니다.** LangChain이 자동 직렬화하기 때문이에요. 차이는 **우리가 짠 Python 코드의 안정성**에 있어요.

이제 5가지 케이스에 이 패턴을 적용해봅시다.


## 7-1. 환율 변환기 → `ExchangeRateResult`

가장 자주 쓰이는 사례. 인자 2개를 받는 도구. **반환값을 Pydantic으로**.


In [8]:
# 정형 반환 스키마 정의 (1교시 5단계 패턴 그대로)
class ExchangeRateResult(BaseModel):
    from_currency: Literal["USD", "EUR", "JPY", "KRW"] = Field(description="출발 통화")
    to_currency: Literal["USD", "EUR", "JPY", "KRW"] = Field(description="도착 통화")
    rate: float = Field(description="환율")

@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> ExchangeRateResult:
    """두 통화 간 환율을 조회합니다.

    Args:
        from_currency: 출발 통화 코드 (USD, EUR, JPY, KRW)
        to_currency: 도착 통화 코드 (USD, EUR, JPY, KRW)
    """
    rates = {("USD","KRW"):1300.0, ("KRW","USD"):1/1300.0,
             ("EUR","KRW"):1400.0, ("JPY","KRW"):9.5}
    rate = rates.get((from_currency, to_currency), 0)
    return ExchangeRateResult(from_currency=from_currency, to_currency=to_currency, rate=rate)

print("✅ 환율 도구 준비 (Pydantic 반환)")

✅ 환율 도구 준비 (Pydantic 반환)


In [16]:
# 도구를 직접 호출해보면 Pydantic 객체로 반환됨
test = get_exchange_rate.invoke({"from_currency": "USD", "to_currency": "KRW"})
print(f"타입:  {type(test).__name__}")
print(f"객체: {test}")
print(f"속성: rate={test.rate}, from={test.from_currency}")

타입:  ExchangeRateResult
객체: from_currency='USD' to_currency='KRW' rate=1300.0
속성: rate=1300.0, from=USD


In [17]:
# LLM과 연결 — LLM 입장에서는 동작 동일
for q in ["100달러는 한국 돈으로 얼마야?",
          "200유로는 원으로 환산하면?",
          "1만엔이면 한국 돈 얼마지?"]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_exchange_rate])}")
    print()

❓ 100달러는 한국 돈으로 얼마야?
💬 100달러는 한국 돈으로 130,000원입니다.

❓ 200유로는 원으로 환산하면?
💬 200 유로는 280,000 원입니다. (환율: 1 유로 = 1,400 원)

❓ 1만엔이면 한국 돈 얼마지?
💬 1만 엔은 약 95,000 원입니다.



**👀 관찰 포인트**

- 도구는 `ExchangeRateResult` 객체를 반환하지만 LLM 응답은 자연어로 잘 나옴
- `Literal` 로 통화 코드를 제한한 덕에 LLM이 잘못된 코드를 만들 수 없음
- 1-2교시의 Pydantic 검증 효과가 도구에도 그대로


## 7-2. 현재 시각·날짜 → `TimeInfo`

인자 없는 도구. 시각과 요일을 함께 묶어 정형 출력.


In [9]:
from datetime import datetime

# 시각 정보 스키마 — 시각·날짜·요일을 한 객체에
class TimeInfo(BaseModel):
    iso_datetime: str = Field(description="ISO 8601 형식 시각")
    date_korean: str = Field(description="한국식 날짜 표현")
    weekday: Literal["월", "화", "수", "목", "금", "토", "일"] = Field(description="요일")

@tool
def get_current_time() -> TimeInfo:
    """현재 한국 시각·날짜·요일을 정형 객체로 반환합니다."""
    now = datetime.now()
    weekday = ["월", "화", "수", "목", "금", "토", "일"][now.weekday()]
    return TimeInfo(
        iso_datetime=now.strftime("%Y-%m-%d %H:%M:%S"),
        date_korean=now.strftime("%Y년 %m월 %d일"),
        weekday=weekday,
    )

print("✅ 시각 도구 준비 (Pydantic 반환)")

✅ 시각 도구 준비 (Pydantic 반환)


In [10]:
# 다양한 시간 관련 질문 — 한 도구로 모두 처리
for q in ["지금 몇 시야?", "오늘 며칠이야?", "오늘 무슨 요일이야?"]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [get_current_time])}")
    print()

❓ 지금 몇 시야?
💬 현재 한국 시각은 2026년 09월 15일 화요일, 09:21입니다.

❓ 오늘 며칠이야?
💬 오늘은 2026년 9월 15일 화요일입니다.

❓ 오늘 무슨 요일이야?
💬 오늘은 화요일입니다.



**👀 관찰 포인트**

- 도구 하나에 **시각·날짜·요일을 모두 담은 정형 객체**를 반환
- LLM이 질문에 맞는 필드만 골라 답변
- 기존 코드 (도구 2개로 분리)보다 깔끔해짐


## 7-3. 상품 검색 → `ProductInfo`

DB 조회 시뮬레이션. **dict 대신 Pydantic 객체**로 결과를 받아  정형 출력 흐름 유지.


In [11]:
# 상품 정보 스키마 (Optional로 검색 실패 표현)
class ProductInfo(BaseModel):
    name: str = Field(description="제품명")
    price: Optional[int] = Field(default=None, description="가격(원), 없으면 None")
    stock: Optional[int] = Field(default=None, description="재고, 없으면 None")
    found: bool = Field(description="검색 성공 여부")

# 가상 상품 DB
PRODUCTS = {
    "iPhone 15":   {"price": 1500000, "stock": 50},
    "Galaxy S24":  {"price": 1200000, "stock": 30},
    "MacBook Pro": {"price": 2800000, "stock": 15},
    "AirPods Pro": {"price": 359000,  "stock": 120},
}

@tool
def search_product(product_name: str) -> ProductInfo:
    """제품 이름으로 상품 정보를 조회합니다.

    Args:
        product_name: 정확한 제품 이름 (예: 'iPhone 15', 'Galaxy S24', 'MacBook Pro', 'AirPods Pro')
    """
    info = PRODUCTS.get(product_name)
    if info is None:
        return ProductInfo(name=product_name, found=False)
    return ProductInfo(name=product_name, price=info["price"], stock=info["stock"], found=True)

print("✅ 상품 검색 도구 준비 (Pydantic 반환)")

✅ 상품 검색 도구 준비 (Pydantic 반환)


In [12]:
for q in ["iPhone 15 가격 알려줘",
          "Galaxy S24 재고 몇 개야?",
          "MacBook Pro 정보 좀 보여줘"]:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, [search_product])}")
    print()

❓ iPhone 15 가격 알려줘
💬 iPhone 15의 가격은 1,500,000원이며, 현재 재고는 50개 있습니다.

❓ Galaxy S24 재고 몇 개야?
💬 Galaxy S24의 재고는 30개입니다. 가격은 1,200,000원입니다.

❓ MacBook Pro 정보 좀 보여줘
💬 MacBook Pro의 정보는 다음과 같습니다:

- **가격**: 2,800,000원
- **재고**: 15개

추가적인 정보가 필요하시면 말씀해 주세요!



**👀 관찰 포인트**

- 검색 실패 시 `found=False` 로 명시적 표현 (dict의 모호한 `{"error": ...}` 보다 안정적)
- `Optional[int]` 패턴이 1교시(이력서 추출)와 동일 — **학습 흐름 연속성**
- 실무에서는 SQL·REST API 결과를 같은 Pydantic 스키마로 받음


## 7-4. 여러 도구 한 번에 등록

지금까지 만든 4개 도구를 **모두 한 LLM에 등록**해봅시다. **반환값이 모두 정형화**되어 있어 일관됨.


In [17]:
# 4개 도구 모두 등록 (multiply는 단순 float 반환, 나머지 3개는 Pydantic)
all_tools = [multiply, get_exchange_rate, get_current_time, search_product]

# 각 도구의 반환 타입 확인
print("📋 등록된 도구의 반환 타입:")
for t in all_tools:
    print(f"   🔧 {t.name:25s} → {t.func.__annotations__.get('return', '?')}")

📋 등록된 도구의 반환 타입:
   🔧 multiply                  → <class 'float'>
   🔧 get_exchange_rate         → <class '__main__.ExchangeRateResult'>
   🔧 get_current_time          → <class '__main__.TimeInfo'>
   🔧 search_product            → <class '__main__.ProductInfo'>


In [18]:
# 다양한 질문 — LLM이 알아서 도구 선택
questions = [
    "지금 몇 시야?",                          # → get_current_time
    "1234 × 5678 계산해줘",                   # → multiply
    "100달러는 원으로 얼마?",                  # → get_exchange_rate
    "AirPods Pro 가격 알려줘",                # → search_product
]

for q in questions:
    print(f"❓ {q}")
    print(f"💬 {run_with_tools(q, all_tools)}")
    print()

❓ 지금 몇 시야?
💬 현재 한국 시각은 2026년 09월 15일 화요일, 09:36입니다.

❓ 1234 × 5678 계산해줘
💬 1234 × 5678의 결과는 7,006,652입니다.

❓ 100달러는 원으로 얼마?
💬 

❓ AirPods Pro 가격 알려줘
💬 AirPods Pro의 가격은 359,000원입니다. 현재 재고는 120개 있습니다.



**👀 관찰 포인트**

- LLM이 질문마다 **4개 중 적절한 도구 하나를 선택**
- Pydantic 정형 반환이 LLM의 동작에 영향 X (자동 직렬화)
- 우리 입장에서 코드는 더 안정적


## 7-5. 도구를 안 써도 되는 질문

LLM은 **필요할 때만** 도구를 씁니다. 일반 대화 질문에는 도구를 호출하지 않아요.


In [19]:
general_questions = [
    "안녕! 너 누구야?",
    "Python의 리스트와 튜플 차이가 뭐야?",
    "행복하게 사는 방법 알려줘",
]

for q in general_questions:
    print(f"❓ {q}")
    answer = run_with_tools(q, all_tools)
    print(f"💬 {answer[:120]}...")
    print()

❓ 안녕! 너 누구야?
💬 안녕하세요! 저는 여러분의 질문에 답하고, 정보를 제공하며, 다양한 작업을 도와주는 AI입니다. 무엇을 도와드릴까요?...

❓ Python의 리스트와 튜플 차이가 뭐야?
💬 Python에서 리스트(List)와 튜플(Tuple)은 모두 여러 값을 저장할 수 있는 데이터 구조이지만, 몇 가지 중요한 차이점이 있습니다.

1. **변경 가능성 (Mutability)**:
   - **리스트*...

❓ 행복하게 사는 방법 알려줘
💬 행복하게 사는 방법은 개인의 가치관과 상황에 따라 다를 수 있지만, 일반적으로 도움이 될 수 있는 몇 가지 방법을 소개할게요:

1. **감사하는 마음 가지기**: 매일 감사한 일을 적어보거나, 주변 사람들에게 감사...



**👀 관찰 포인트**

- 도구를 4개 등록해뒀지만 **하나도 호출 안 됨**
- 헬퍼 함수 내부의 `if not response.tool_calls: return response.content` 가 작동
- LLM이 똑똑하게 **"이건 도구 없이도 답할 수 있다"** 고 판단

이게 Tool Calling의 핵심 매력입니다.


---

# Step 8. 🎯 도전 과제

본인이 관심 있는 도구를 직접 만들어보세요. **Pydantic 정형 반환 패턴**을 적용하면서.

```python
class MyResult(BaseModel):
    field1: str
    field2: int

@tool
def my_tool(arg: type) -> MyResult:
    """..."""
    return MyResult(field1="...", field2=42)
```

## 과제 A: 단위 변환기 (실무 빈도 ★★★★★)

길이·무게·온도 변환 도구. 반환값을 `ConversionResult` Pydantic 객체로.

**힌트**
```python
class ConversionResult(BaseModel):
    original_value: float
    original_unit: str
    converted_value: float
    converted_unit: str

@tool
def convert_length(value: float, from_unit: str, to_unit: str) -> ConversionResult:
    """길이 단위를 변환합니다.

    Args:
        value: 변환할 값
        from_unit: 출발 단위 ('m', 'cm', 'km', 'inch', 'ft')
        to_unit: 도착 단위 ('m', 'cm', 'km', 'inch', 'ft')
    """
    # 변환 로직
    ...
```

테스트: "180cm가 몇 미터야?", "10km는 몇 마일이야?"

---

## 과제 B: 날짜 계산기 (실무 빈도 ★★★★☆)

```python
class DateDiff(BaseModel):
    date1: str
    date2: str
    days: int
    weeks: int

@tool
def days_between(date1: str, date2: str) -> DateDiff:
    """두 날짜(YYYY-MM-DD) 사이의 일수·주수를 반환합니다."""
    ...
```

---

## 과제 C: 가상 날씨 API (실무 빈도 ★★★★☆)

```python
class WeatherInfo(BaseModel):
    city: str
    temperature_c: float
    condition: Literal["맑음", "흐림", "비", "눈"]
    humidity_percent: int

@tool
def get_weather(city: str) -> WeatherInfo:
    """도시의 현재 날씨를 조회합니다."""
    ...
```

테스트: "서울이랑 부산 날씨 알려줘"   ← 도구를 두 번 호출!



---

## 과제 D: docstring 일부러 짧게 써보기 (실험 ★★★★★)

같은 도구의 docstring을 **자세한 버전 vs 짧은 버전**으로 만들어 비교.

```python
@tool
def add_v1(a: int, b: int) -> int:
    """더하기"""        # ← 너무 짧음
    return a + b

@tool
def add_v2(a: int, b: int) -> int:
    """두 정수의 합을 계산합니다.
    수학 계산이 필요한 모든 경우에 사용하세요."""
    return a + b
```

같은 질문에 LLM이 어떤 차이를 보이는지 관찰하세요.

---

> 💡 **하나만 골라서 깊이.** Pydantic 정형 반환 패턴이 손에 익도록.


---

## 오늘 배운 빌드업

```
 chain = prompt | llm | parser   (LLM 출력을 정형화)
                ↓
[+ 추가 ①] @tool 데코레이터
[+ 추가 ②] bind_tools
[+ 추가 ③] tool_calls 확인
[+ 추가 ④] 도구 실행 + 2차 호출 (ToolMessage)
[+ 추가 ⑤] ⭐ 도구 반환값도 Pydantic으로 정형화
                ↓
[헬퍼 함수] run_with_tools(question, tools)
```

## OutputParser·Pydantic 학습의 일관 사용

 Pydantic 사용처 |
| --- | --- |
| LLM 답변 정형화 (`PydanticOutputParser`) |
| 함수 반환값 정형화 (`AlertAction`, `LogRecord`) |
 | **도구 반환값 정형화** (`ExchangeRateResult`, `TimeInfo`, `ProductInfo`) ⭐ |

LLM 호출이든, 함수 결과든, 도구 반환이든 **일관되게 Pydantic으로** 받으면 코드 전체가 안정됩니다.

## 핵심 컴포넌트 5가지

| 컴포넌트 | 역할 |
| --- | --- |
| `@tool` | 함수를 LangChain 도구로 변환 |
| `bind_tools()` | LLM에 도구 등록 |
| `tool_calls` | LLM이 호출하려는 도구 정보 |
| `ToolMessage` | 도구 실행 결과를 LLM에 전달 |
| **`BaseModel` 반환** | 도구 결과를 정형 객체로 ⭐ |

## 오늘 만든 도구 4개

| 도구 | 반환 타입 |
| --- | --- |
| `multiply` | `float` |
| `get_exchange_rate` | `ExchangeRateResult` |
| `get_current_time` | `TimeInfo` |
| `search_product` | `ProductInfo` |

